# CS604 Project - 00 Setup and data check

Runs on a **Colab GPU runtime** (VS Code: Select Kernel -> Colab). Run top to bottom.
Everything is stored on the Colab VM's local disk (`/content`), which is wiped on disconnect.
Step 2 (download + extract) runs again in every new session, but is skipped if the data is already there.

## 1. GPU and disk

In [ ]:
!nvidia-smi
!df -h /content

## 2. Download and extract (about 44 GB)

Each archive is streamed straight into `tar`, so download and extraction overlap and no 44 GB archive is stored.
Skipped automatically if the data is already prepared in this session (marker file `.ready`).
The marker is only written if everything succeeds. If the connection drops, just re-run the cell (the archives restart).

In [ ]:
%%bash
set -e
set -o pipefail
D=/content/cs604-course-data
U=https://huggingface.co/datasets/doem1997/cs604-course-data/resolve/main
if [ -f $D/.ready ]; then echo "Data already prepared - skipping download."; exit 0; fi
mkdir -p $D && cd $D
curl -sSfL $U/train.txt -o train.txt
curl -sSfL $U/val.txt -o val.txt
# stream both archives in parallel, extracting as they download
curl -sSfL $U/val.tar.gz   | tar -xz &
P1=$!
curl -sSfL $U/train.tar.gz | tar -xz
wait $P1
touch .ready
df -h /content
ls $D

## 3. Labels: counts and class balance

Only `train.txt` has labels. `val.txt` is image paths only (labels are withheld, scored by CodaBench).

In [ ]:
import os, collections
ROOT = "/content/cs604-course-data"

def read_labels(name):                      # "path class_id" per line (train only)
    with open(f"{ROOT}/{name}") as f:
        return [(p, int(c)) for p, c in (l.split() for l in f)]

def read_paths(name):                       # "path" per line (val: no labels)
    with open(f"{ROOT}/{name}") as f:
        return [l.strip() for l in f if l.strip()]

train, val = read_labels("train.txt"), read_paths("val.txt")
print("train:", len(train), " val:", len(val))
print("examples:", train[:2], val[:2])

cnt = collections.Counter(c for _, c in train)
print(f"train: {len(cnt)} classes, images/class min={min(cnt.values())} max={max(cnt.values())}, "
      f"class ids {min(cnt)}..{max(cnt)}")

## 4. Do the files exist? Image sizes

In [ ]:
import random
from PIL import Image

random.seed(0)
chk = [p for p, _ in random.sample(train, 5000)] + random.sample(val, 2000)   # random sample, a full scan is slow
missing = [p for p in chk if not os.path.exists(f"{ROOT}/{p}")]
print("missing files (of 7000 sampled):", len(missing), missing[:3])

random.seed(0)
sample = random.sample(train, 300)
sizes = [Image.open(f"{ROOT}/{p}").size for p, _ in sample]
ws, hs = zip(*sizes)
print(f"width  min/median/max: {min(ws)} / {sorted(ws)[len(ws)//2]} / {max(ws)}")
print(f"height min/median/max: {min(hs)} / {sorted(hs)[len(hs)//2]} / {max(hs)}")
print("distinct sizes in sample:", len(set(sizes)))
print("mean file size (KB):", sum(os.path.getsize(f"{ROOT}/{p}") for p, _ in sample) / len(sample) / 1024)
print("CPU cores:", os.cpu_count())

## 5. Look at a few images

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, (p, c) in zip(axes.ravel(), random.sample(train, 12)):
    ax.imshow(Image.open(f"{ROOT}/{p}").convert("RGB")); ax.set_title(f"class {c}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()